# DICOM to BIDS Conversion — Batch (`heudiconv`)

This notebook runs Phase 2 heudiconv conversion (DICOM → BIDS NIfTIs) across multiple participants.

**Before using this notebook:**
- Complete Phase 1 in `heudiconv_single.ipynb` to create your `heuristic.py`
- Test Phase 2 on at least one subject in `heudiconv_single.ipynb` to confirm the output is correct

**Two execution modes are provided:**
- **Option A — Bash loop:** Runs each subject sequentially in this notebook. Good for small datasets.
- **Option B — Slurm:** Generates job scripts and submits them to the HPC cluster in parallel. Recommended for large datasets.

#### History
- 9/9/21 dcosme — initial code
- Refactored for CNLab pipeline documentation

## 1. Imports

In [ ]:
import os

## 2. Set Project Variables

Edit these variables before running either execution mode below.

> **Geoscan reference:** `project = 'geoscan_v2'`, subjects followed `GEO###`, DICOMs were at `/raw/geoscan/T2/{subject}_T2/*.dcm`, session label `t2` or `t3`.

In [ ]:
# ── Project paths ─────────────────────────────────────────────────────────────
project     = 'your_project'          # Project folder under /data00/projects/
project_dir = f'/data00/projects/{project}'
bids_dir    = os.path.join(project_dir, 'data/bids_data')
heuristic   = os.path.join(project_dir, 'scripts/BIDS/heudiconv/code/heuristic.py')

# ── Raw DICOM path pattern ────────────────────────────────────────────────────
# {subject} is replaced automatically per subject in the loop below.
# Geoscan example: /raw/geoscan/T2/{subject}_T2/*.dcm
raw_dicom_pattern = '/raw/your_project/{subject}/*/*.dcm'

# ── Singularity image ─────────────────────────────────────────────────────────
heudiconv_sif = '/data00/tools/singularity_images/heudiconv_0.8.0'

# ── Session label ─────────────────────────────────────────────────────────────
session = 't1'   # Session label (e.g. 't1', 't2', 'baseline')

# ── Subject list ──────────────────────────────────────────────────────────────
# List subject IDs exactly as they appear in the DICOM folder names
subs = [
    'sub-001',
    'sub-002',
    'sub-003',
    # add more subjects here...
]

print(f"Project      : {project}")
print(f"Session      : {session}")
print(f"Subjects ({len(subs)}): {subs}")
print(f"heuristic.py : {'EXISTS' if os.path.exists(heuristic) else 'NOT FOUND — complete Phase 1 first'}")

---
## Option A: Bash Loop (Run in Notebook)

Generates a `.job` shell script for each subject and immediately executes it with `bash`. Subjects run sequentially. Use this for small datasets or when the cluster is unavailable.

Completed subjects are skipped if their BIDS output directory already exists, so re-running after a failure is safe.

In [ ]:
# Job script directory for Option A
job_dir = os.path.join(project_dir, 'scripts/BIDS/jobs')
os.makedirs(job_dir, exist_ok=True)

# Template for the heudiconv bash job
# {ID} is replaced with the subject ID; {subject} is kept as-is for heudiconv's own substitution
bash_job_template = r"""#!/bin/bash
singularity run --cleanenv \\
    -B /data00/projects/{PROJECT}:/base \\
    -B /fmriDataRaw/fmri_data_raw:/raw \\
    {SIF} \\
    -d {DICOM_PATTERN} \\
    -o /base/data/bids_data/ \\
    -f /base/scripts/BIDS/heudiconv/code/heuristic.py \\
    -ss {SESSION} -s {ID} -c dcm2niix -b --overwrite
"""

print(f"Job scripts will be written to: {job_dir}")

In [ ]:
# Loop through subjects, create job scripts, and run them
failed_subs = []

for s in subs:
    sub_bids_dir = os.path.join(bids_dir, f'sub-{s}', f'ses-{session}')

    # Skip if already converted
    if os.path.exists(sub_bids_dir):
        print(f'--- Skipping {s}: BIDS output already exists ---')
        continue

    print(f'\n{"="*60}')
    print(f'Converting: {s}')
    print(f'{"="*60}')

    # Write the job script for this subject
    job_script = bash_job_template.format(
        PROJECT=project,
        SIF=heudiconv_sif,
        DICOM_PATTERN=raw_dicom_pattern,
        SESSION=session,
        ID=s,
        subject='{subject}'  # keep as literal for heudiconv
    )

    file_path = os.path.join(job_dir, f'heudiconv_{s}.job')
    with open(file_path, 'w') as f:
        f.write(job_script)

    # Execute the job script
    result = os.system(f'bash {file_path}')

    if result != 0:
        print(f'!! ERROR: heudiconv failed for {s} (exit code {result})')
        failed_subs.append(s)
    else:
        print(f'Completed: {s}')

# Summary
print(f'\n{"="*60}')
print(f'Batch complete. Attempted: {len(subs)}, Failed: {len(failed_subs)}')
if failed_subs:
    print(f'Failed subjects: {failed_subs}')

---
## Option B: Slurm (HPC Cluster)

Generates `.job` files with Slurm headers and prints the `sbatch` commands to run on the cluster. SSH to the cluster node and paste the printed commands to submit all subjects in parallel.

Adjust `#SBATCH` resource settings (time, CPUs, memory) based on your cluster's requirements and dataset size.

In [ ]:
# Slurm job output directory
slurm_dir = os.path.join(project_dir, 'scripts/BIDS/jobs/heudiconv')
os.makedirs(slurm_dir, exist_ok=True)
os.makedirs(os.path.join(slurm_dir, 'out'), exist_ok=True)  # for .out and .err logs

# Template for the Slurm job script
slurm_job_template = r"""#!/bin/bash
#SBATCH --job-name=heudiconv_{ID}
#SBATCH --output=out/heudiconv_{ID}.out
#SBATCH --error=out/heudiconv_{ID}.err
#SBATCH --time=02:00:00
#SBATCH --cpus-per-task=8

srun singularity run --cleanenv \\
    -B /data00/projects/{PROJECT}:/base \\
    -B /fmriDataRaw/fmri_data_raw:/raw \\
    {SIF} \\
    -d {DICOM_PATTERN} \\
    -o /base/data/bids_data/ \\
    -f /base/scripts/BIDS/heudiconv/code/heuristic.py \\
    -ss {SESSION} -s {ID} -c dcm2niix -b --overwrite
"""

print(f"Slurm job scripts will be written to: {slurm_dir}")

In [ ]:
# Generate one Slurm job file per subject
for s in subs:
    job_content = slurm_job_template.format(
        ID=s,
        PROJECT=project,
        SIF=heudiconv_sif,
        DICOM_PATTERN=raw_dicom_pattern,
        SESSION=session,
        subject='{subject}'  # keep as literal for heudiconv
    )

    job_path = os.path.join(slurm_dir, f'heudiconv_{s}.job')
    with open(job_path, 'w') as f:
        f.write(job_content)

    print(f'Written: {job_path}')

print(f'\nAll {len(subs)} job scripts written to: {slurm_dir}')

In [ ]:
# Print the sbatch commands to paste into the cluster terminal
#
# To submit:
#   1. SSH to the Slurm master node: ssh <username>@asc.upenn.edu@cls000
#   2. Copy and paste the output below into the terminal

print(f'cd {slurm_dir}\n')
for s in subs:
    print(f'sbatch -D {slurm_dir} heudiconv_{s}.job')

---
## Next Steps

After conversion is complete for all subjects:

1. Run `fieldmap_intendedfor.ipynb` to add `IntendedFor` fields to fieldmap JSON sidecars
2. Validate the dataset with the [BIDS Validator](https://bids-standard.github.io/bids-validator/)
3. Proceed to fMRIPrep